# Первый опыт классификации: название, аннотация или оба поля?

Сравним простой классификатор **TF-IDF + логистическая регрессия** с правилом «всегда самый частый класс». Для каждого используем один и тот же заранее проверенный набор статей и одно разделение на обучающую и валидационную выборки.

Это предварительный опыт с ручной разметкой, **не проверка H2** и не результат ruPhysBERT. В исходном пакете пока нет утверждённых меток, а инструкция по пяти классам ещё требует согласования. Поэтому обычный запуск сейчас проверяет входные файлы и **пропускает обучение**. Метки не выводятся из названий журналов, ключевых слов или кодов автоматически.

Ноутбук не скачивает статьи, не меняет реестры и не создаёт допуск к обучению. Для реального опыта нужны отдельно согласованные данные, метки и группы связанных публикаций. Формальная тестовая выборка здесь не используется.

Зависимости: установи `requirements-notebooks.txt` в окружение проекта и выбери его ядром ноутбука. Выполняй ячейки сверху вниз. Сохраняемые результаты находятся только в локальном `artifacts/experiments/`; перед добавлением ноутбука в Git нужно очистить выводы ячеек.

In [ ]:
"""Предварительное сравнение способов представления текста для классификации."""

from __future__ import annotations

import hashlib
import json
import platform
import unicodedata
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.pipeline import Pipeline

ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / 'src' / 'collect').is_dir() and (directory / 'README.md').is_file()),
    None,
)

if ROOT is None:
    raise RuntimeError('Открой ноутбук из каталога проекта или notebooks/.')

CANDIDATES_PATH = ROOT / 'manifests/results/h2_candidates_jinr_clean_20261010_v2/cleaned_candidates.jsonl'
CANDIDATES_SHA256 = 'ba5eab6ff76a34e02a50e245d095b565362d7ad1289a59b1abe74d98a92d2ae7'
ANNOTATIONS_PATH = ROOT / 'data/processed/experiments/classification_pilot/labels.jsonl'
REVIEW_PATH = ROOT / 'data/processed/experiments/classification_pilot/review.json'
OUTPUT_ROOT = ROOT / 'artifacts/experiments/classification_baseline'
NOTEBOOK_PATH = ROOT / 'notebooks/01_classification_baseline.ipynb'
NOTEBOOK_CODE_SHA256 = hashlib.sha256('\n\n'.join(
    ''.join(cell['source'])
    for cell in json.loads(NOTEBOOK_PATH.read_text(encoding='utf-8'))['cells']
    if cell['cell_type'] == 'code'
).encode('utf-8')).hexdigest()
CLASS_CODES = ['C1', 'C2', 'C3', 'C4', 'C5']
SEED = 42
VARIANTS = {
    'title': 'Название',
    'abstract': 'Аннотация',
    'title_abstract': 'Название + аннотация',
}
VECTORIZER_PARAMETERS: dict[str, Any] = {
    'ngram_range': (1, 2),
    'min_df': 1,
    'max_features': 40_000,
    'sublinear_tf': True,
}
CLASSIFIER_PARAMETERS: dict[str, Any] = {
    'C': 1.0,
    'class_weight': 'balanced',
    'max_iter': 2_000,
    'random_state': SEED,
}
# В scikit-learn 1.9.1 анализатор выводит str из значения по умолчанию
# zero_division='warn', хотя API также принимает числовой 0.
METRIC_PARAMETERS: dict[str, Any] = {'zero_division': 0}


## Что потребуется для реального запуска

1. Согласовать с научруком названия и границы пяти классов; `C1`–`C5` здесь — только коды, не уже утверждённые рубрики. Независимо разметить выбранные статьи и разобрать разногласия. Пробные примеры, по которым уточнялась инструкция, не должны попадать в валидационную выборку.
2. Проверить связанные версии, совпадения текстов и признаки-подсказки (ГРНТИ, PACS, названия рубрик). Назначить каждой группе только одну выборку. Разделение готовится **до** сравнения моделей и не подбирается под результат.
3. Подготовить `data/processed/experiments/classification_pilot/labels.jsonl`: по одной записи на статью, без копирования текстов. Поля: `candidate_id` из сохранённого пакета, `label` из `C1`–`C5`, `group_id`, `split` (`train` или `validation`). Обе выборки должны содержать все пять классов.
4. Рядом сохранить `review.json`, связывающий проверку с точными файлами. Его поля приведены ниже. Пока пункты не выполнены, значения проверок остаются `false`; простая смена флажков не заменяет проверку и согласование использования. Это запись о конкретном опыте, не изменение рабочего реестра прав.

Пример структуры одной строки разметки (**не готовая метка**):
```json
{"candidate_id": "jinr:<идентификатор>", "label": "C1", "group_id": "group-001", "split": "train"}
```

Структура `review.json` (файл намеренно не создаётся автоматически):
```json
{
  "schema_version": 1,
  "experiment_kind": "exploratory_manual_labels",
  "candidate_snapshot_sha256": "SHA-256 из ячейки настроек",
  "annotations_sha256": "SHA-256 точных байтов labels.jsonl",
  "classes": {"C1": "название", "C2": "название", "C3": "название", "C4": "название", "C5": "название"},
  "reviewed_by": "кто проверил",
  "approval_reference": "основание согласования использования именно этих данных",
  "label_guide_reviewed": false,
  "labels_reviewed": false,
  "groups_reviewed": false,
  "leakage_reviewed": false,
  "practice_examples_excluded": false,
  "use_approved": false
}
```

Классификатор видит только выбранное текстовое поле: идентификаторы, журнал, URL, авторы и метки не используются как признаки. Размеры выборок и число групп выводятся перед обучением. Малый опыт полезен для отладки, но не заменяет расчёт размера выборки и полноценную проверку H2.

In [ ]:
def _json_object(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    """Отклонить повторяющиеся ключи вместо тихой подмены значения."""

    result: dict[str, Any] = {}

    for key, value in pairs:
        if key in result:
            raise ValueError(f'Повторяющийся ключ JSON: {key}')

        result[key] = value

    return result


def _reject_constant(value: str) -> None:
    """Отклонить нечисловые константы вне стандарта JSON."""

    raise ValueError(f'Недопустимая константа JSON: {value}')


def _read_json(text: str) -> dict[str, Any]:
    """Прочитать один объект JSON со строгой проверкой ключей."""

    result = json.loads(text, object_pairs_hook=_json_object, parse_constant=_reject_constant)

    if not isinstance(result, dict):
        raise ValueError('Ожидался объект JSON.')

    return result


def _read_jsonl(path: Path) -> list[dict[str, Any]]:
    """Прочитать записи JSONL и сообщить номер ошибочной строки."""

    records: list[dict[str, Any]] = []

    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        try:
            records.append(_read_json(line))

        except ValueError as exception:
            raise ValueError(f'{path.name}, строка {line_number}: {exception}') from exception

    return records


def _normalize_text(text: str) -> str:
    """Нормализовать текст только для проверки точных совпадений."""

    return ' '.join(unicodedata.normalize('NFKC', text).casefold().split())


def prepare_dataset(
    candidate_records: list[dict[str, Any]],
    label_records: list[dict[str, Any]],
    review: dict[str, Any],
    annotations_sha256: str,
) -> pd.DataFrame:
    """Проверить согласование, метки и отсутствие явных утечек между выборками."""

    if review.get('schema_version') != 1 or review.get('experiment_kind') != 'exploratory_manual_labels':
        raise ValueError('Нужна проверка версии 1 именно для опыта с ручными метками.')

    if review.get('candidate_snapshot_sha256') != CANDIDATES_SHA256:
        raise ValueError('Проверка относится к другому снимку кандидатов.')

    if review.get('annotations_sha256') != annotations_sha256:
        raise ValueError('Файл меток изменился после проверки.')

    for field in (
        'label_guide_reviewed', 'labels_reviewed', 'groups_reviewed',
        'leakage_reviewed', 'practice_examples_excluded', 'use_approved',
    ):
        if review.get(field) is not True:
            raise ValueError(f'Не подтверждена проверка: {field}.')

    for field in ('reviewed_by', 'approval_reference'):
        if not isinstance(review.get(field), str) or not review[field].strip():
            raise ValueError(f'Не заполнено поле проверки: {field}.')

    classes = review.get('classes')

    if not isinstance(classes, dict) or set(classes) != set(CLASS_CODES):
        raise ValueError('Нужны согласованные названия всех пяти классов C1–C5.')

    if any(not isinstance(name, str) or not name.strip() for name in classes.values()):
        raise ValueError('Название каждого класса должно быть непустой строкой.')

    if len({_normalize_text(name) for name in classes.values()}) != len(CLASS_CODES):
        raise ValueError('Названия классов не должны повторяться.')

    candidates = pd.DataFrame(candidate_records)
    labels = pd.DataFrame(label_records)
    candidate_columns = {'candidate_id', 'title_clean', 'abstract_clean'}
    label_columns = {'candidate_id', 'label', 'group_id', 'split'}

    if not candidate_columns.issubset(candidates.columns) or set(labels.columns) != label_columns:
        raise ValueError('Некорректные поля кандидатов или разметки; проверь контракт выше.')

    if labels.empty or candidates['candidate_id'].duplicated().any() or labels['candidate_id'].duplicated().any():
        raise ValueError('Нужны непустые метки и уникальные candidate_id в каждом файле.')

    for column in label_columns:
        if not all(
            isinstance(value, str) and bool(value) and value == value.strip()
            for value in labels[column].tolist()
        ):
            raise ValueError(f'Все значения {column} должны быть непустыми строками без крайних пробелов.')

    if not set(labels['label']).issubset(CLASS_CODES) or set(labels['split']) != {'train', 'validation'}:
        raise ValueError('Допустимы только C1–C5 и обе выборки train/validation.')

    unknown_ids = set(labels['candidate_id']) - set(candidates['candidate_id'])

    if unknown_ids:
        raise ValueError(f'В снимке отсутствуют {len(unknown_ids)} размеченных статей.')

    ambiguous_flags = {'multiple_title_variants', 'multiple_abstract_variants'}
    ambiguous_ids = {
        record['candidate_id'] for record in candidate_records
        if ambiguous_flags.intersection(record.get('source_review_flags', []))
    }

    if set(labels['candidate_id']).intersection(ambiguous_ids):
        raise ValueError('У выбранных карточек неоднозначные варианты текста; сначала разреши их отдельно.')

    dataset = labels.merge(candidates[list(candidate_columns)], on='candidate_id', validate='one_to_one')
    dataset = dataset.rename(columns={'title_clean': 'title', 'abstract_clean': 'abstract'})

    for column in ('title', 'abstract'):
        if not all(isinstance(value, str) and bool(value.strip()) for value in dataset[column].tolist()):
            raise ValueError(f'Для честного сравнения всем статьям требуется поле {column}.')

        if dataset[column].str.contains(r'(?i)\b(?:УДК|UDC|PACS|ГРНТИ|МРНТИ)\s*[:;]?\s*\d', regex=True).any():
            raise ValueError(f'В {column} остались классификационные коды — возможная утечка.')

    dataset['title_abstract'] = dataset['title'].str.strip() + '\n' + dataset['abstract'].str.strip()

    if (dataset.groupby('group_id')['split'].nunique() > 1).any():
        raise ValueError('Связанные статьи из одной группы попали в разные выборки.')

    if (dataset.groupby('group_id')['label'].nunique() > 1).any():
        raise ValueError('Связанные версии одной работы имеют противоречащие метки.')

    for variant in VARIANTS:
        normalized = dataset[variant].map(_normalize_text)
        overlap = dataset.assign(normalized=normalized).groupby('normalized')

        if (overlap['label'].nunique() > 1).any():
            raise ValueError(f'Одинаковые тексты {variant} имеют противоречащие метки.')

        if (overlap['split'].nunique() > 1).any() or (overlap['group_id'].nunique() > 1).any():
            raise ValueError(f'Совпадения {variant} требуют объединения групп и проверки разделения.')

        if variant == 'title_abstract' and normalized.duplicated().any():
            raise ValueError('Полные дубликаты увеличивают вес одной статьи; оставь по одной записи.')

    for split_name in ('train', 'validation'):
        if set(dataset.loc[dataset['split'] == split_name, 'label']) != set(CLASS_CODES):
            raise ValueError(f'В выборке {split_name} представлены не все пять классов.')

    return dataset.sort_values('candidate_id').reset_index(drop=True)


In [ ]:
dataset: pd.DataFrame | None = None
review: dict[str, Any] | None = None
annotations_sha256: str | None = None
review_sha256: str | None = None

if not CANDIDATES_PATH.is_file():
    print('Обучение пропущено: локальный снимок кандидатов не найден.')

elif hashlib.sha256(CANDIDATES_PATH.read_bytes()).hexdigest() != CANDIDATES_SHA256:
    raise ValueError('Снимок кандидатов изменился; не подменяем входные данные незаметно.')

else:
    candidate_records = _read_jsonl(CANDIDATES_PATH)
    print(f'Снимок проверен: {len(candidate_records)} кандидатов, не готовых меток.')
    missing_paths = [path for path in (ANNOTATIONS_PATH, REVIEW_PATH) if not path.is_file()]

    if missing_paths:
        print('Обучение пропущено. Сначала нужны согласованные метки и проверка разделения:')

        for missing_path in missing_paths:
            print(f'  {missing_path.relative_to(ROOT)}')

    else:
        annotations_sha256 = hashlib.sha256(ANNOTATIONS_PATH.read_bytes()).hexdigest()
        review_sha256 = hashlib.sha256(REVIEW_PATH.read_bytes()).hexdigest()
        review = _read_json(REVIEW_PATH.read_text(encoding='utf-8'))
        dataset = prepare_dataset(candidate_records, _read_jsonl(ANNOTATIONS_PATH), review, annotations_sha256)
        display(pd.crosstab(dataset['label'], dataset['split']).reindex(CLASS_CODES))
        print('Групп по выборкам:', dataset.groupby('split')['group_id'].nunique().to_dict())


## Модели и оценка

TF-IDF превращает слова и пары слов в численные признаки. Словарь и веса IDF обучаются **только на обучающей выборке** внутри `Pipeline`; валидационная выборка лишь преобразуется. Параметры одинаковы для трёх вариантов текста, подбор параметров в этом опыте не проводится.

Логистическая регрессия обучается по этим признакам. `class_weight='balanced'` учитывает дисбаланс только в обучающей выборке. `DummyClassifier` всегда выбирает её самый частый класс — это нижний ориентир, а не полезная модель.

Главная метрика — **macro-F1**: среднее F1 по пяти классам, каждый класс имеет одинаковый вес. Дополнительно посмотрим F1 каждого класса, матрицы ошибок и сами ошибки. Это описательные результаты одного валидационного разделения: малые различия не объявляем статистически значимыми. Связанные версии учитываем группами, но похожие тексты и подсказки в них требуют ручной проверки.

In [ ]:
def run_baselines(dataset: pd.DataFrame, class_codes: list[str]) -> dict[str, Any]:
    """Сравнить три представления на одном заранее проверенном разделении."""

    train = dataset.loc[dataset['split'] == 'train']
    validation = dataset.loc[dataset['split'] == 'validation']

    if train.empty or validation.empty or len(class_codes) < 2:
        raise ValueError('Нужны обе непустые выборки и хотя бы два класса.')

    score_rows: list[dict[str, Any]] = []
    class_rows: list[dict[str, Any]] = []
    prediction_rows: list[dict[str, Any]] = []
    fitted_models: dict[str, Pipeline] = {}

    for variant in VARIANTS:
        pipeline = Pipeline([
            ('tfidf', TfidfVectorizer(**VECTORIZER_PARAMETERS)),
            ('classifier', LogisticRegression(**CLASSIFIER_PARAMETERS)),
        ])
        pipeline.fit(train[variant].tolist(), train['label'].tolist())
        fitted_models[variant] = pipeline
        dummy = DummyClassifier(strategy='most_frequent')
        dummy.fit(np.zeros((len(train), 1)), train['label'].tolist())
        predictions_by_model = {
            'TF-IDF + LogisticRegression': pipeline.predict(validation[variant].tolist()),
            'Dummy': dummy.predict(np.zeros((len(validation), 1))),
        }

        for model_name, predictions in predictions_by_model.items():
            score_rows.append({
                'input': variant, 'model': model_name,
                'macro_f1': float(f1_score(validation['label'], predictions, labels=class_codes, average='macro', **METRIC_PARAMETERS)),
                'train_size': len(train), 'validation_size': len(validation),
            })
            report = classification_report(
                validation['label'], predictions, labels=class_codes, output_dict=True, **METRIC_PARAMETERS,
            )

            if not isinstance(report, dict):
                raise TypeError('classification_report должен вернуть словарь показателей.')

            for class_code in class_codes:
                class_metrics = report[class_code]

                if not isinstance(class_metrics, dict):
                    raise TypeError(f'Не получены показатели класса {class_code}.')

                class_rows.append({'input': variant, 'model': model_name, 'class': class_code, **class_metrics})

            for record, predicted_label in zip(validation.to_dict('records'), predictions, strict=True):
                prediction_rows.append({
                    'candidate_id': record['candidate_id'], 'group_id': record['group_id'],
                    'input': variant, 'model': model_name,
                    'expected': record['label'], 'predicted': str(predicted_label),
                })

    return {
        'scores': pd.DataFrame(score_rows),
        'per_class': pd.DataFrame(class_rows),
        'predictions': pd.DataFrame(prediction_rows),
        'models': fitted_models,
    }


results: dict[str, Any] | None = None

if dataset is None:
    print('Реальные модели не обучались: входные данные опыта пока не готовы.')

else:
    results = run_baselines(dataset, CLASS_CODES)
    display(results['scores'].round({'macro_f1': 4}))
    display(results['per_class'].round(4))


In [ ]:
if results is not None and dataset is not None and review is not None:
    figure, axes = plt.subplots(1, 3, figsize=(16, 4))

    for axis, (variant, caption) in zip(axes, VARIANTS.items(), strict=True):
        predictions = results['predictions']
        selected = predictions.loc[(predictions['input'] == variant) & (predictions['model'] != 'Dummy')]
        ConfusionMatrixDisplay.from_predictions(
            selected['expected'], selected['predicted'], labels=CLASS_CODES,
            ax=axis, colorbar=False, cmap='Blues',
        )
        axis.set_title(caption)
        axis.set_xlabel('Предсказанный класс')
        axis.set_ylabel('Проверенная метка')

    figure.tight_layout()
    plt.show()
    print('Названия классов:', review['classes'])
    predictions = results['predictions']
    errors = predictions.loc[(predictions['model'] != 'Dummy') & (predictions['expected'] != predictions['predicted'])]
    errors = errors.merge(dataset[['candidate_id', 'title']], on='candidate_id', validate='many_to_one')
    display(errors.head(15))


## Сохранение результата

Если реальный опыт выполнен, сохраняем метрики, предсказания и описание запуска в новом локальном каталоге `artifacts/experiments/classification_baseline/`. Исходные тексты, обученные модели и случайно полученные искусственные метрики здесь не публикуются. Сохраняются хеши входных файлов и кода, версия Python, версии библиотек и параметры. В `selection.jsonl` остаётся точная выборка с идентификаторами, группами, метками и разделением — её можно восстановить, даже если рабочий файл разметки впоследствии изменится.

После запуска сформулируем вывод: какой вариант лучше на **этой** валидационной выборке, какие классы путаются и какие ограничения есть у разметки. Только затем имеет смысл сравнивать с ruBERT и ruSciBERT на том же проверенном наборе.

In [ ]:
if results is not None and dataset is not None and review is not None:
    run_time = datetime.now(timezone.utc)
    output_dir = OUTPUT_ROOT / run_time.strftime('%Y%m%dT%H%M%S_%fZ')
    output_dir.mkdir(parents=True, exist_ok=False)

    for result_name in ('scores', 'per_class', 'predictions'):
        results[result_name].to_csv(output_dir / f'{result_name}.csv', index=False)

    selection_path = output_dir / 'selection.jsonl'
    dataset[['candidate_id', 'group_id', 'label', 'split']].to_json(
        selection_path, orient='records', lines=True, force_ascii=False,
    )

    run_metadata = {
        'experiment_kind': 'exploratory_manual_labels',
        'formal_h2_result': False,
        'created_at': run_time.isoformat(),
        'candidate_snapshot_sha256': CANDIDATES_SHA256,
        'annotations_sha256': annotations_sha256,
        'review_sha256': review_sha256,
        'notebook_code_sha256': NOTEBOOK_CODE_SHA256,
        'selection_sha256': hashlib.sha256(selection_path.read_bytes()).hexdigest(),
        'python_version': platform.python_version(),
        'classes': review['classes'],
        'seed': SEED,
        'vectorizer_parameters': VECTORIZER_PARAMETERS,
        'classifier_parameters': CLASSIFIER_PARAMETERS,
        'metric_parameters': METRIC_PARAMETERS,
        'dummy_strategy': 'most_frequent',
        'variants': VARIANTS,
        'split_counts': dataset['split'].value_counts().to_dict(),
        'group_counts': dataset.groupby('split')['group_id'].nunique().to_dict(),
        'versions': {name: version(name) for name in ('numpy', 'pandas', 'scikit-learn', 'matplotlib')},
    }
    (output_dir / 'run.json').write_text(json.dumps(run_metadata, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(f'Локальные результаты: {output_dir.relative_to(ROOT)}')


## Отдельная техническая проверка

Следующая ячейка проверяет только исполнение моделей на маленьких **выдуманных** строках. Она не использует статьи, не заполняет `dataset`/`results`, не сохраняет метрики и ничего не говорит о качестве классификации физики. Слово-маркер встречается только в искусственной валидационной выборке: проверяем, что оно не попало в обученный словарь TF-IDF.

In [ ]:
def smoke_test_pipeline() -> dict[str, Any]:
    """Проверить исполнение на искусственных строках, не оценивая реальные статьи."""

    synthetic_rows: list[dict[str, Any]] = []
    marker = 'проверочныймаркер'

    for class_index, class_code in enumerate(CLASS_CODES):
        for sample_index in range(6):
            split_name = 'train' if sample_index < 4 else 'validation'
            extra_text = marker if split_name == 'validation' else 'учебнаястрока'
            title = f'Искусственная тема{class_index} образец{sample_index} {extra_text}'
            abstract = f'Выдуманное описание тема{class_index} пример{sample_index} {extra_text}'
            synthetic_rows.append({
                'candidate_id': f'synthetic-{class_index}-{sample_index}',
                'group_id': f'synthetic-group-{class_index}-{sample_index}',
                'split': split_name, 'label': class_code,
                'title': title, 'abstract': abstract, 'title_abstract': title + '\n' + abstract,
            })

    synthetic_results = run_baselines(pd.DataFrame(synthetic_rows), CLASS_CODES)

    if len(synthetic_results['scores']) != 6 or len(synthetic_results['predictions']) != 60:
        raise AssertionError('Не все сочетания моделей и представлений выполнены.')

    for fitted_model in synthetic_results['models'].values():
        if marker in fitted_model.named_steps['tfidf'].vocabulary_:
            raise AssertionError('Валидационный маркер попал в обученный словарь.')

    return {'status': 'ok', 'data': 'synthetic_only', 'variants': 3, 'models_per_variant': 2}


smoke_test_pipeline()
